In [ ]:
from datetime import timedelta

from catboost import CatBoostRegressor
import numpy as np
import optuna
import pandas as pd

In [ ]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_train = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv", parse_dates=["visit_date"])
df_test = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv", parse_dates=["visit_date"])

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

In [ ]:
categorical_features = [
    "air_store_id",
    "air_genre_name",
    "day_of_week",
    "month",
    "day_pattern",
    "prefecture",
]

numeric_features = [
    "doy_cos",
    "store_mean_cum",
    "store_dow_mean_cum",
    "store_roll_mean_14",
    "store_roll_mean_28",
    "genre_geo_mean_cum",
    "reserve_visitors",
    "walk_in_ratio",
]

binary_features = ["is_off_day"]

features = categorical_features + numeric_features + binary_features

X_full = df_train[features]
y_full = np.log1p(df_train["visitors"].values)

In [ ]:
def get_custom_cv_splits(df: pd.DataFrame, n_splits: int = 3, val_days: int = 39) -> list:
    splits = []
    max_date = df["visit_date"].max()

    for i in range(n_splits):
        val_end = max_date - timedelta(days=i * val_days)
        val_start = val_end - timedelta(days=val_days - 1)

        train_mask = df["visit_date"] < val_start
        val_mask = (df["visit_date"] >= val_start) & (df["visit_date"] <= val_end)

        train_idx = df.index[train_mask].tolist()
        val_idx = df.index[val_mask].tolist()

        splits.append((train_idx, val_idx))
        print(
            f"Fold {i + 1}: Train ends {val_start - timedelta(days=1):%Y-%m-%d}"
            + f"| Val: {val_start:%Y-%m-%d} to {val_end:%Y-%m-%d}"
        )

    return splits[::-1]


cv_splits = get_custom_cv_splits(df_train, n_splits=3, val_days=39)

In [ ]:
best_iters_overall = []


def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 5, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
    }

    cv_scores = []
    fold_iters = []

    for train_idx, val_idx in cv_splits:
        X_train, y_train = X_full.iloc[train_idx], y_full[train_idx]
        X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]

        model = CatBoostRegressor(**params, cat_features=categorical_features)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

        cv_scores.append(model.get_best_score()["validation"]["RMSE"])
        fold_iters.append(model.get_best_iteration())

    trial.set_user_attr("mean_best_iter", int(np.mean(fold_iters)))

    return np.mean(cv_scores)


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("\n--- BEST PARAMS ---")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"\n best rmsle during CV: {study.best_value:.4f}")

optimal_iterations = study.best_trial.user_attrs["mean_best_iter"]
print(f"Optimal Iter: {optimal_iterations}")

In [10]:
final_params = study.best_params.copy()
final_params["iterations"] = optimal_iterations
final_params["loss_function"] = "RMSE"
final_params["eval_metric"] = "RMSE"
final_params["random_seed"] = 42

final_model = CatBoostRegressor(**final_params, cat_features=categorical_features)

final_model.fit(X_full, y_full, verbose=100)

0:	learn: 0.7885320	total: 25.9ms	remaining: 4.33s
100:	learn: 0.5163574	total: 2.17s	remaining: 1.44s
167:	learn: 0.5089904	total: 3.62s	remaining: 0us


CatBoostRegressor(bagging_temperature=0.1800668160458957, cat_features=['air_store_id', 'air_genre_name', 'day_of_week', 'month', 'day_pattern', 'prefecture'], depth=10, eval_metric='RMSE', iterations=168, l2_leaf_reg=1.339794018760351, learning_rate=0.04454603954683453, loss_function='RMSE', random_seed=42, random_strength=0.4012042608993718)

In [11]:
X_test = df_test[features]

preds_log = final_model.predict(X_test)

preds_real_clipped = np.clip(np.expm1(preds_log), 1.0, None)

submission = pd.DataFrame(
    {
        "id": df_test["air_store_id"] + "_" + df_test["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real_clipped,
    }
)

submission_path = "submission_catboost_optuna.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,3.994039
1,air_08cb3c4ee6cd6a22_2017-04-23,14.391397
2,air_f8233ad00755c35c_2017-04-23,7.432667
3,air_234d3dbf7f3d5a50_2017-04-23,6.989904
4,air_a563896da3777078_2017-04-23,26.164973
